In [103]:
import pandas as pd
import datetime
import numpy as np
from copy import deepcopy
from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from catboost import CatBoostClassifier, Pool

In [114]:
def uplift_fit_predict_catboost(model, X_train, treatment_train, target_train, X_test, cat_features):
    """
    :param model: 
    :param X_train: 
    :param treatment_train: 
    :param target_train: 
    :param X_test: 
    :return: 
    """
    X_train.assign(treatment=treatment_train.values)
    _X_train = Pool(X_train, target_train, cat_features=cat_features)
    _model = deepcopy(model).fit(_X_train, verbose=100, early_stopping_rounds=200)
    predict_treatment = _model.predict_proba(X_test.assign(treatment = 1))[:, 1]
    predict_control = _model.predict_proba(X_test.assign(treatment = 0))[:, 1]
    predict_uplift = predict_treatment - predict_control
    return predict_uplift



def uplift_score(prediction, treatment, target, rate=0.3):
    """
    Подсчет Uplift Score
    """
    order = np.argsort(-prediction)
    treatment_n = int((treatment == 1).sum() * rate)
    treatment_p = target[order][treatment[order] == 1][:treatment_n].mean()
    control_n = int((treatment == 0).sum() * rate)
    control_p = target[order][treatment[order] == 0][:control_n].mean()
    score = treatment_p - control_p
    return score

In [110]:
df_train = pd.read_csv('./data/data/uplift_train.csv', index_col='client_id')
df_test = pd.read_csv('./data/data/uplift_test.csv', index_col='client_id')
dataset = pd.read_parquet('./data/preprocessed/dataset_v0.parquet')
df_train = df_train.merge(
    dataset,
    left_index=True,
    right_index=True,
    how='left'
)
df_test = df_test.merge(
    dataset,
    left_index=True,
    right_index=True,
    how='left'
)
cat_features = ['gender', 'age_oon', 'issue_redeem_delay_oon']

In [116]:
model = CatBoostClassifier(
    loss_function='Logloss',
    eval_metric='AUC',
    learning_rate=0.05,
    iterations=1000,
    depth=5,
    random_strength=0,
    l2_leaf_reg=0.5,
    task_type='GPU',
    random_seed=42,
    verbose=True
)

In [112]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 200039 entries, 000012768d to fffff6ce77
Data columns (total 32 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   treatment_flg                 200039 non-null  int64  
 1   target                        200039 non-null  int64  
 2   age                           200039 non-null  int64  
 3   gender                        200039 non-null  object 
 4   first_issue_unixtime          200039 non-null  float64
 5   first_redeem_unixtime         200039 non-null  float64
 6   issue_redeem_delay            200039 non-null  float64
 7   age_oon                       200039 non-null  bool   
 8   issue_redeem_delay_oon        200039 non-null  bool   
 9   transaction_id_nunique        200039 non-null  int64  
 10  regular_points_received_sum   200039 non-null  float64
 11  regular_points_received_mean  200039 non-null  float64
 12  express_points_received_sum   200039

In [117]:
indices_train = df_train.index
indices_test = df_test.index
indices_learn, indices_valid = train_test_split(df_train.index, test_size=0.3, random_state=123)

valid_uplift = uplift_fit_predict_catboost(
    model=model,
    X_train=df_train.loc[indices_learn, :].drop(columns=['treatment_flg', 'target']),
    treatment_train=df_train.loc[indices_learn, 'treatment_flg'],
    target_train=df_train.loc[indices_learn, 'target'],
    X_test=df_train.loc[indices_valid, :].drop(columns=['treatment_flg', 'target']),
    cat_features=cat_features
)
valid_score = uplift_score(
    valid_uplift,
    treatment=df_train.loc[indices_valid, 'treatment_flg'].values,
    target=df_train.loc[indices_valid, 'target'].values,
)
print('Validation score:', valid_score)

Default metric period is 5 because AUC is/are not implemented for GPU


0:	total: 18.5ms	remaining: 18.4s
100:	total: 992ms	remaining: 8.83s
200:	total: 1.96s	remaining: 7.8s
300:	total: 2.88s	remaining: 6.68s
400:	total: 3.77s	remaining: 5.64s
500:	total: 4.67s	remaining: 4.65s
600:	total: 5.55s	remaining: 3.69s
700:	total: 6.47s	remaining: 2.76s
800:	total: 7.38s	remaining: 1.83s
900:	total: 8.28s	remaining: 910ms
999:	total: 9.18s	remaining: 0us
Validation score: 0.03218312235012133
